In [17]:
#from google.colab import drive
#drive.mount('/content/drive')

In [18]:
%pip install -U torch transformers datasets evaluate accelerate scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [19]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

import evaluate
import numpy as np

import re

In [20]:
temp = load_dataset("csv", data_files="data/Sentiment140.csv", encoding="ISO-8859-1")["train"]
temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
temp = temp.filter(lambda x: x["Label"] in [0, 4])

def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

def replace(text):
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'http\S+|www\.\S+', 'HTTPURL', text)
    return text

def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]] }

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=1)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=1)

dataset_dict = {
    "train": temp["train"].shuffle(seed=1).select(range(100)),
    "test": test_valid["test"].select(range(50)),
    "validation": test_valid["train"].select(range(50)),
}

dataset_dict

{'train': Dataset({
     features: ['Label', 'Text'],
     num_rows: 100
 }),
 'test': Dataset({
     features: ['Label', 'Text'],
     num_rows: 50
 }),
 'validation': Dataset({
     features: ['Label', 'Text'],
     num_rows: 50
 })}

In [21]:
from transformers import AutoTokenizer

model_path = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

In [22]:
MAX_LENGTH = 128

def preprocess_function(batch):
    return tokenizer(batch["Text"], truncation=True, padding=False, max_length=MAX_LENGTH)

tokenized = {}
for split, ds in dataset_dict.items():
    ds = ds.rename_column("Label", "labels")
    tokenized[split] = ds.map(preprocess_function, batched=True, remove_columns=["Text"])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [23]:
import torch
import torch.nn as nn
from transformers import RobertaModel, RobertaPreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput

POOLING = "mean"  # "mean", "cls", "max", "meanmax"

class RobertaForSequenceClassificationWithPooling(RobertaPreTrainedModel):
    def __init__(self, config, pooling: str = "mean"):
        super().__init__(config)
        self.num_labels = config.num_labels

        self.config.pooling = getattr(config, "pooling", pooling)
        self.pooling = self.config.pooling

        self.roberta = RobertaModel(config, add_pooling_layer=False)

        hidden_size = config.hidden_size
        in_features = hidden_size * 2 if self.pooling == "meanmax" else hidden_size

        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(in_features, config.num_labels)

        self.post_init()

    def _mean_pool(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-9)
        return summed / denom

    def _max_pool(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).bool()
        masked = last_hidden_state.masked_fill(~mask, torch.finfo(last_hidden_state.dtype).min)
        return masked.max(dim=1).values

    def _pool(self, last_hidden_state, attention_mask):
        if self.pooling == "cls":
            return last_hidden_state[:, 0]
        if self.pooling == "max":
            return self._max_pool(last_hidden_state, attention_mask)
        if self.pooling == "meanmax":
            return torch.cat(
                [
                    self._mean_pool(last_hidden_state, attention_mask),
                    self._max_pool(last_hidden_state, attention_mask),
                ],
                dim=1,
            )
        return self._mean_pool(last_hidden_state, attention_mask)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        pooled = self._pool(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

model = RobertaForSequenceClassificationWithPooling.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    pooling=POOLING,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassificationWithPooling LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task

In [24]:
FREEZE_BASE_MODEL = True  # set True to train only pooling+classifier

if FREEZE_BASE_MODEL:
    for p in model.roberta.parameters():
        p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Pooling: {model.config.pooling}")
print(f"Trainable parameters: {trainable:,} / {total:,} ({trainable / total:.2%})")

Pooling: mean
Trainable parameters: 1,538 / 134,310,914 (0.00%)


In [25]:
import os

RESUME_CHECKPOINT = "/content/drive/MyDrive/outputs/pooling-bertweet/checkpoint-45000"

# resume_from_checkpoint = None
resume_from_checkpoint = RESUME_CHECKPOINT if (RESUME_CHECKPOINT and os.path.isdir(RESUME_CHECKPOINT)) else None

if resume_from_checkpoint:
    print(f"Resuming from checkpoint: {resume_from_checkpoint}")
else:
    print("No checkpoint found (or not set). Training from scratch.")

No checkpoint found (or not set). Training from scratch.


In [26]:
from transformers import TrainingArguments, Trainer
from transformers.utils.notebook import NotebookProgressCallback

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/outputs/pooling-bertweet",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.remove_callback(NotebookProgressCallback)

trainer.train(resume_from_checkpoint=resume_from_checkpoint)
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results.get("eval_accuracy"))
print("Test F1:", results.get("eval_f1"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Brian\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Brian\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Test accuracy: 0.62
Test F1: 0.6545454545454545


In [27]:
final_dir = "/content/drive/MyDrive/outputs/pooling-bertweet/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved to:", final_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/outputs/pooling-bertweet/final


In [28]:
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from transformers.utils.notebook import NotebookProgressCallback
import evaluate
import numpy as np

model_dir = "/content/drive/MyDrive/outputs/pooling-bertweet/final"

tokenizer = AutoTokenizer.from_pretrained(model_dir, use_fast=True)
model = RobertaForSequenceClassificationWithPooling.from_pretrained(model_dir)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

eval_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/outputs/eval-pooling-bertweet",
    per_device_eval_batch_size=64,
    report_to="none",
    do_train=False,
    do_eval=True,
)

trainer = Trainer(
    model=model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.remove_callback(NotebookProgressCallback)

results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Pooling:", getattr(model.config, "pooling", "<unknown>"))
print("Test accuracy:", results.get("eval_accuracy"))
print("Test F1:", results.get("eval_f1"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

c:\Users\Brian\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Pooling: mean
Test accuracy: 0.62
Test F1: 0.6545454545454545
